# What Makes a Popular Song?
## A First Pandas Project with 32,000 Spotify Tracks

*Module 3 Practice Project - Pandas Basics*


## Project Background

Every day Spotify decides which songs to push to millions of listeners. Each
track gets a **popularity score** (0-100), and each song also has measurable
audio traits: how **danceable** it is, how much **energy** it has, its tempo,
duration, and more.

In this project you play the role of a music-market analyst. A record label
wants to know: what do popular songs have in common, and do different genres
(pop, rap, rock, latin, r&b, edm) behave differently?

You will use only the Pandas tools from Module 3 - loading, inspecting,
cleaning, filtering, grouping, and reshaping. No charts and no statistics yet.


## Project Objective

> **Clean up a messy music dataset and use it to describe what popular songs
> look like, compare the six genres, and find the standout tracks.**

Every task below answers one small question on the way to that goal.


## Dataset

Source: *Spotify Songs*, published via the TidyTuesday project
(github.com/rfordatascience/tidytuesday - originally compiled from the Spotify
API).

| File | Contents |
|---|---|
| `datasets/spotify_tracks.csv` | ~32,800 tracks: name, artist, popularity (0-100), genre, subgenre, audio features (danceability, energy, tempo, duration...), key (0-11) and mode (0/1) |
| `datasets/key_notes_lookup.csv` | Small lookup table translating the `key` number into a musical note name |

**Known quirks:** the same track can appear on several playlists; a handful of
rows are missing names; some values look impossible. You will find all of this.


## Workflow

1. Load and inspect
2. Check quality (missing values, duplicates, impossible values)
3. Clean
4. Make codes readable (map + merge)
5. Filter interesting groups of songs
6. Create useful new columns
7. Compare genres with groupby and pivot tables
8. Export a clean final dataset


In [ ]:
import numpy as np
import pandas as pd

# Section 1 - First Look


### Task 1 - Open the Dataset

**Problem**

Before any analysis, confirm what actually arrived: how many songs, which columns.

**Your Task**

Load `datasets/spotify_tracks.csv` into a DataFrame called `tracks`. Show the number of rows and columns, then view the first and last five rows.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`pd.read_csv()`, `.shape`, `.head()`, `.tail()`.

</details>

**Pseudocode**

```
START
Read the CSV file
Print the shape
Show first and last rows
END
```

### Task 2 - Structure Check

**Problem**

Column types must match what they hold, and missing values must be visible early.

**Your Task**

Run `.info()`. Which columns have missing values? Which columns hold numbers vs text?

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`.info()` prints dtype and non-null counts per column in one call.

</details>

**Pseudocode**

```
START
Print structural summary
Note columns with missing values
END
```

### Task 3 - Number Sanity Check

**Problem**

Audio features should live inside sensible ranges (e.g., danceability between 0 and 1). Summary statistics expose anything weird instantly.

**Your Task**

Run `.describe()` on the numeric columns. Look at min and max of every column: which values look impossible or extreme? Note them for cleaning.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Check especially `tempo` (can a song have zero tempo?) and `duration_ms`.

</details>

**Pseudocode**

```
START
Summarise numeric columns
Compare min/max against plausible ranges
List suspicious values
END
```

### Task 4 - Which Genres Are In Here?

**Problem**

The six playlist genres are the backbone of every comparison later. Know their sizes before trusting averages.

**Your Task**

Show how many songs belong to each `playlist_genre`, sorted from most to least.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`.value_counts()` already sorts by frequency.

</details>

**Pseudocode**

```
START
Count songs per genre
Display sorted counts
END
```

# Section 2 - Quality Check and Cleaning


### Task 5 - Missing Values

**Problem**

Rows without a track name or artist cannot be reported to the label.

**Your Task**

Count missing values per column and convert to percentages. Which three columns are affected?

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`.isnull().sum()` for counts; divide by `len(df)` for percentages.

</details>

**Pseudocode**

```
START
Count nulls per column
Compute percentage of total rows
Report affected columns
END
```

### Task 6 - Duplicate Songs

**Problem**

The same song can be added to several playlists - so the same `track_id` may appear many times. For song-level analysis those repeats would inflate counts and skew averages.

**Your Task**

First count fully identical rows (`.duplicated().sum()`). Then count repeated `track_id` values using `.duplicated(subset=['track_id'])`. Display a few rows belonging to one duplicated `track_id` so you can SEE the repetition. Record both numbers - you will need them next task.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Pick one duplicated id with `.value_counts()` or a boolean mask, then filter the DataFrame with that mask to display all its copies.

</details>

**Pseudocode**

```
START
Count exact duplicate rows
Count duplicated track_id values
Show all rows of one duplicated track_id
Record both counts
END
```

### Task 7 - Deduplicate With Intent

**Problem**

Which copy of a repeated song should survive? The one from its most relevant home: keeping the copy with the highest popularity is a defensible rule.

**Your Task**

Sort the data by `track_popularity` descending, then drop duplicates on `track_id` keeping the first. Report rows before and after. Confirm zero repeated ids remain.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`.sort_values(...)` then `.drop_duplicates(subset=["track_id"], keep="first")` - the sort-then-keep pattern from Class 12.

</details>

**Pseudocode**

```
START
Sort by popularity descending
Drop duplicates on track_id keeping the best copy
Verify no repeated track_ids remain
END
```

### Task 8 - Impossible Values

**Problem**

Task 3 flagged suspicious numbers. A tempo of exactly 0 is not a song; a track shorter than 30 seconds is likely noise data.

**Your Task**

Count rows where `tempo == 0` and rows where `duration_ms` is below 30000 (30 seconds). Remove these rows from your working DataFrame and confirm the row count dropped accordingly.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Build a boolean mask with `|`, then filter: `df = df[~mask]`.

</details>

**Pseudocode**

```
START
Mask rows with tempo equal to zero OR duration under 30 seconds
Count them
Remove them from the DataFrame
Confirm new shape
END
```

### Task 9 - Handle the Missing Names

**Problem**

Only about five rows lack a track name. Dropping them loses nothing meaningful; imputing a song title makes no sense.

**Your Task**

Justify (in one comment line) why dropping is right here, then drop rows where `track_name` is missing. Verify no nulls remain anywhere.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`.dropna(subset=[...])` targets specific columns only.

</details>

**Pseudocode**

```
START
Drop rows missing track_name
Verify no missing values remain
END
```

# Section 3 - Making Codes Readable


### Task 10 - Map Mode Numbers to Words

**Problem**

`mode` is stored as 0/1. Nobody outside the data team knows what that means - labels make every future output self-explanatory.

**Your Task**

Create a dictionary mapping `{0: "Minor", 1: "Major"}` and use `.map()` to build a new column `mode_name`. Show the value counts of the new column as proof.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`s.map(dict)` recodes every element through the dictionary (Class 15).

</details>

**Pseudocode**

```
START
Define the mode dictionary
Map it onto a new mode_name column
Show value counts
END
```

### Task 11 - Merge in the Musical Notes Lookup

**Problem**

`key` is stored as a number 0-11. The lookup file translates each number to a note name (C, F#, B...). Joining lookup tables is everyday analyst work - but only safe if the key is unique.

**Your Task**

Load `datasets/key_notes_lookup.csv`. Prove its `key` column has no duplicates. Then left-merge it onto your tracks on `key`, enable `indicator=True`, and verify the row count is unchanged and every row matched.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`.duplicated().sum()` on the lookup key; then `pd.merge(tracks, lookup, how="left", on="key", indicator=True)` and check `_merge` value counts.

</details>

**Pseudocode**

```
START
Load the lookup table
Verify key uniqueness
Left-merge onto tracks with indicator enabled
Check row count and merge status
END
```

# Section 4 - Asking Questions with Filters


### Task 12 - Party Playlist Candidates

**Problem**

The label wants high-danceability pop songs for a party campaign.

**Your Task**

Select pop songs (`playlist_genre == "pop"`) with `danceability` above 0.8. Keep only these columns: track_name, track_artist, danceability, energy, popularity. Sort by danceability descending and show the top 10.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Boolean mask combined with `.loc[rows, [cols]]`; chain `.sort_values(ascending=False).head(10)`.

</details>

**Pseudocode**

```
START
Filter pop AND danceability above 0.8
Keep the four reporting columns via .loc
Sort by danceability descending
Show top 10
END
```

### Task 13 - Three Genres Side by Side

**Problem**

Comparing rap, rock, and r&b requires isolating exactly those categories - `isin()` handles multi-value membership cleanly.

**Your Task**

Filter tracks whose genre is rap, rock, or r&b. Report the number of songs found per genre with `value_counts()`.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`df[df["playlist_genre"].isin([...])]`.

</details>

**Pseudocode**

```
START
Filter genres using isin
Count rows per genre
END
```

### Task 14 - The Middle Class of Popularity

**Problem**

Viral hits and forgotten songs distort averages. The 'solid mid-tier' (popularity 50 to 70 inclusive) is where careers are built - isolate it with a range test.

**Your Task**

Use `between()` to keep popularity from 50 to 70. What share of all tracks falls in this band? (Divide its size by the total.)

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`between(50, 70)` is inclusive on both ends; wrap the mask sum over `len(df)`.

</details>

**Pseudocode**

```
START
Keep popularity between 50 and 70
Compute share of total tracks
END
```

### Task 15 - Find the Love Songs

**Problem**

A radio station wants ballads. Song titles containing 'love' are the quickest proxy - string search does this in one line.

**Your Task**

Find tracks whose name contains 'love' (case-insensitive: search lower-cased names or use a pattern covering both cases). How many exist? Show five examples with artist names.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`.str.contains("love", na=False)` on `track_name.str.lower()` - recall why `na=False` matters.

</details>

**Pseudocode**

```
START
Search lowercase track names for the word love
Count matches
Show five example rows with artists
END
```

### Task 16 - One Readable Query

**Problem**

Complex questions deserve readable code. Query syntax lets us write almost plain English.

**Your Task**

Using ONE `.query()` call: energetic rock songs (`energy > 0.9`) released onto playlists with `loudness` above -5 that are longer than 240000 ms. Return track_name, artist, energy, loudness sorted by energy descending.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Inside query strings you can write `and` directly (unlike boolean masks).

</details>

**Pseudocode**

```
START
Write one query combining genre, energy, loudness, duration
Sort result by energy descending
END
```

# Section 5 - New Columns That Answer Questions


### Task 17 - Song Length in Minutes

**Problem**

Nobody thinks in milliseconds. Convert `duration_ms` to minutes in a new column `duration_min` (divide by 60000). Then report: average song length overall, plus the longest and shortest remaining tracks.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Arithmetic on a Series creates a new column directly; `.max()` / `.min()` locate extremes.

</details>

**Pseudocode**

```
START
Create duration_min column
Report mean, max and min
END
```

### Task 18 - Energy Labels

**Problem**

Categorical buckets communicate faster than decimals: calling a track 'high energy' means something to a label executive; 0.93 does not.

**Your Task**

Add an `energy_level` column: 'high' when energy > 0.8, otherwise 'normal'. Use `np.where` (Module 2) or a small function with `.apply()` - your choice. Show value counts afterwards.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`np.where(condition, "high", "normal")` vectorises the decision in one call.

</details>

**Pseudocode**

```
START
Apply threshold logic to energy column
Store result as energy_level
Show value counts
END
```

# Section 6 - Comparing Genres


### Task 19 - Genre League Table

**Problem**

Which genre wins on popularity, and which gets people dancing? One aggregation table answers both.

**Your Task**

Group by `playlist_genre` and compute in one `.agg()` call: average popularity, average danceability, average energy, and song count. Sort by average popularity descending.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Named aggregation: `.agg(new_col=("source_col", "mean"), ...)` keeps output clean.

</details>

**Pseudocode**

```
START
Group by genre
Aggregate the three averages plus count with named outputs
Sort by average popularity
END
```

### Task 20 - Do Major and Minor Sound Different?

**Problem**

Producers argue that minor-key songs dominate certain genres. Test the claim by splitting each genre into major vs minor.

**Your Task**

Group by `playlist_genre` and `mode_name` together, computing average popularity and song count. Read the table: within which genre is the major-vs-minor popularity gap largest?

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Multi-key groupby: pass a list `.groupby(["playlist_genre", "mode_name"])`. Optionally `.unstack()` to place Minor/Major side by side.

</details>

**Pseudocode**

```
START
Group by genre and mode_name
Aggregate average popularity and count
Compare gaps across genres
END
```

### Task 21 - Pivot Table: Genre x Energy Level

**Problem**

A two-way summary (genre rows, energy-level columns) is exactly what becomes a grouped bar chart next module - build it now with totals included.

**Your Task**

Build a `pd.pivot_table`: index = genre, columns = `energy_level`, values = popularity, aggfunc = mean, margins = True.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`pd.pivot_table(df, values=, index=, columns=, aggfunc=, margins=True)`.

</details>

**Pseudocode**

```
START
Pivot genres against energy levels
Average popularity per cell
Enable margins for totals
END
```

### Task 22 - Every Song vs Its Genre Average

**Problem**

Popularity means different things in different genres - a 60 in rock is not a 60 in pop. Dividing each song's popularity by its own genre's average creates a fair, cross-genre comparison score. Because the result must land back on every row (not collapse per group), this is a job for `transform`.

**Your Task**

Broadcast each genre's average popularity back to all rows with `groupby(...).transform("mean")` (verify the length matches). Divide popularity by it into a new column `pop_vs_genre`. Find the five most over-performing tracks in the whole dataset.

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

`len(series) == len(df)` proves row preservation; `.nlargest(5, "pop_vs_genre")` ranks the result.

</details>

**Pseudocode**

```
START
Broadcast genre-average popularity to every row
Verify lengths match
Divide popularity by the broadcast average
Rank top five over-performers
END
```

### Task 23 - Hall of Fame and Hall of Shame

**Problem**

Stakeholders always ask for extremes: the ten biggest hits and the ten least popular surviving tracks.

**Your Task**

Use `nlargest(10, ...)` and `nsmallest(10, ...)` on popularity to produce both lists (name, artist, genre, popularity).

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Both methods were demonstrated in Class 13; they return the n extreme rows pre-sorted.

</details>

**Pseudocode**

```
START
Extract ten most popular tracks
Extract ten least popular tracks
END
```

# Section 7 - Wrap-Up


### Task 24 - Export the Clean Dataset

**Problem**

The cleaned table is the deliverable that later modules (visualisation, statistics) will consume.

**Your Task**

Final checks on your working DataFrame: shape, `info()`, zero missing values, zero repeated track_ids. Then export it as `spotify_tracks_clean.csv` (without the index).

<details>
<summary><b>Hint / Guidance</b> (click to expand)</summary>

Re-run your earlier checks quickly, then `.to_csv(path, index=False)`.

</details>

**Pseudocode**

```
START
Verify shape, dtypes, nulls and duplicates
Export to CSV without index
END
```

### Task 25 - Key Findings

Replace the prompts below with YOUR observations, citing the task that produced each:

1. The most popular genre on average is ___, and the most danceable is ___ (Task 19).
2. About ___% of all songs sit in the popularity middle-class 50-70 (Task 14).
3. The biggest major-vs-minor popularity gap occurs in ___ (Task 20).
4. Data removed during cleaning: ___ duplicates, ___ impossible rows, ___ missing-name rows (Tasks 7-9).



## Next Stage

This cleaned dataset will power the coming modules:

- **Visualisation (Module 4):** the Task 21 pivot becomes a grouped bar chart; genre averages become bar charts; popularity distributions become histograms.
- **Statistics & EDA (Module 5):** outlier detection on duration and tempo, distribution shapes of danceability and energy.
- **Time patterns:** this dataset has no dates - time-series analysis arrives with a different project later in the course.
